In [0]:
from pyspark.sql.functions import *
from datetime import *
# Get batch_id from widget
dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id")
print(batch_id)

101


In [0]:
# Read metadata for this batch
master_tbl_df = spark.sql(f"SELECT * FROM metadata.master_tble WHERE batch_id = '{batch_id}'")
display(master_tbl_df)

batch_id,source_name,data_load_strategy,source_tbl_name,bronze_tbl_name,silver_tbl_name,watermark_column
101,azure_sql,DeltaLoad_SCD1,dbo.sales,bronze.sales,silver.sales,2000-09-07T15:23:08.535Z


In [0]:
row=master_tbl_df.first()
src_tbl_name_dynamic=row["source_tbl_name"]
bronze_tbl_name_dynamic=row["bronze_tbl_name"]
silver_tbl_name_dynamic=row["silver_tbl_name"]
data_load_strategy=row["data_load_strategy"]
prev_modified_date_dynamic=row["watermark_column"]

print("Source:",src_tbl_name_dynamic)
print("Bronze:",bronze_tbl_name_dynamic)
print("Silver:",silver_tbl_name_dynamic)
print("Strategy:",data_load_strategy)
print("Prev watermark:",prev_modified_date_dynamic)

Source: dbo.sales
Bronze: bronze.sales
Silver: silver.sales
Strategy: DeltaLoad_SCD1
Prev watermark: 2000-09-07 15:23:08.535000


In [0]:
#jdbc connection
src_url=(
    f"jdbc:sqlserver://dm-project1.database.windows.net:1433;"
    f"database=Sales_db;"
    f"user=dmuser;"
    f"password=Dipti123@+;"
)

In [0]:
#Fromat watermark for sql Server
try:
    dt=datetime.strptime(str(prev_modified_date_dynamic),'%Y-%m-%d %H:%M:%S.%f')
    formatted_date=dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
except ValueError:
    raise ValueError("Invalid watermark format")

In [0]:
#QUE: Why dont we used if condition here?
#Read source incrementally
filter_condition=f"ModifiedDate>'{formatted_date}'"
source_df=(
    spark.read.format("jdbc")
    .option("url",src_url)
    .option("dbtable",f"(SELECT * FROM {src_tbl_name_dynamic} WHERE {filter_condition})AS temp")
    .load()
)
display(source_df)

salesOrderID,orderDate,customerID,orderValue,ModifiedDate
1001,2025-11-10T00:00:00.000Z,1,2500.00,2025-11-10T00:00:00.000Z
1002,2025-11-11T00:00:00.000Z,2,1500.50,2025-11-11T00:00:00.000Z
1003,2025-11-12T00:00:00.000Z,3,3200.75,2025-11-12T00:00:00.000Z
1004,null,1,500.00,2025-11-13T00:00:00.000Z
1006,2025-11-15T00:00:00.000Z,99,1000.00,2025-11-15T00:00:00.000Z
1007,2025-11-16T00:00:00.000Z,4,750.25,2025-11-16T00:00:00.000Z
1008,2025-11-17T00:00:00.000Z,5,1800.00,2025-11-17T00:00:00.000Z
1009,2025-11-18T00:00:00.000Z,6,2200.00,2025-11-18T00:00:00.000Z
1010,2025-11-19T00:00:00.000Z,7,3000.00,2025-11-19T00:00:00.000Z


In [0]:
# Compute new watermark
src_max_modified_date = source_df.agg(max("ModifiedDate")).first()[0]
print("New watermark:", src_max_modified_date)

New watermark: 2025-11-19 00:00:00


In [0]:
if src_max_modified_date is None:
    print("No new data to process")
elif data_load_strategy=="DeltaLoad_SCD1":
    # "Deduplicate+audit"
    source_df_add_col=source_df.dropDuplicates().withColumn("load_date",current_timestamp())

    #write bronze
    source_df_add_col.write.mode("overwrite").saveAsTable(bronze_tbl_name_dynamic)

    #write into Silver
    silver_query=f"""
    MERGE INTO {silver_tbl_name_dynamic} AS target
    USING {bronze_tbl_name_dynamic} AS source
    ON source.salesOrderId=target.salesOrderId
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """ 

    spark.sql(silver_query)

    #update watermark
    spark.sql(f"""
          UPDATE metadata.master_tble
          SET watermark_column='{src_max_modified_date}'
          WHERE batch_id='{batch_id}'
          """)
    
else:
    print("Unsupported strategy:",data_load_strategy)


In [0]:
%sql
SELECT * FROM bronze.sales;

salesOrderID,orderDate,customerID,orderValue,ModifiedDate,load_date
1008,2025-11-17T00:00:00.000Z,5,1800.00,2025-11-17T00:00:00.000Z,2025-11-30T09:35:59.759Z
1010,2025-11-19T00:00:00.000Z,7,3000.00,2025-11-19T00:00:00.000Z,2025-11-30T09:35:59.759Z
1002,2025-11-11T00:00:00.000Z,2,1500.50,2025-11-11T00:00:00.000Z,2025-11-30T09:35:59.759Z
1009,2025-11-18T00:00:00.000Z,6,2200.00,2025-11-18T00:00:00.000Z,2025-11-30T09:35:59.759Z
1003,2025-11-12T00:00:00.000Z,3,3200.75,2025-11-12T00:00:00.000Z,2025-11-30T09:35:59.759Z
1007,2025-11-16T00:00:00.000Z,4,750.25,2025-11-16T00:00:00.000Z,2025-11-30T09:35:59.759Z
1004,null,1,500.00,2025-11-13T00:00:00.000Z,2025-11-30T09:35:59.759Z
1001,2025-11-10T00:00:00.000Z,1,2500.00,2025-11-10T00:00:00.000Z,2025-11-30T09:35:59.759Z
1006,2025-11-15T00:00:00.000Z,99,1000.00,2025-11-15T00:00:00.000Z,2025-11-30T09:35:59.759Z


In [0]:
%sql
SELECT * FROM silver.sales;

salesOrderID,orderDate,customerID,orderValue,ModifiedDate,load_date
1008,2025-11-17T00:00:00.000Z,5,1800.00,2025-11-17T00:00:00.000Z,2025-11-30T09:35:59.759Z
1010,2025-11-19T00:00:00.000Z,7,3000.00,2025-11-19T00:00:00.000Z,2025-11-30T09:35:59.759Z
1002,2025-11-11T00:00:00.000Z,2,1500.50,2025-11-11T00:00:00.000Z,2025-11-30T09:35:59.759Z
1009,2025-11-18T00:00:00.000Z,6,2200.00,2025-11-18T00:00:00.000Z,2025-11-30T09:35:59.759Z
1003,2025-11-12T00:00:00.000Z,3,3200.75,2025-11-12T00:00:00.000Z,2025-11-30T09:35:59.759Z
1007,2025-11-16T00:00:00.000Z,4,750.25,2025-11-16T00:00:00.000Z,2025-11-30T09:35:59.759Z
1004,null,1,500.00,2025-11-13T00:00:00.000Z,2025-11-30T09:35:59.759Z
1001,2025-11-10T00:00:00.000Z,1,2500.00,2025-11-10T00:00:00.000Z,2025-11-30T09:35:59.759Z
1006,2025-11-15T00:00:00.000Z,99,1000.00,2025-11-15T00:00:00.000Z,2025-11-30T09:35:59.759Z
